# 10 — Final Model Robustness and Sensitivity Analysis

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

**Purpose:** Challenge the Stage-08 enhanced-static Random Forest and Stage-09 temporal findings *before* selecting a final production model. This notebook does **not** train the final production model or generate a national map.

**Important:** Spatially held-out performance measures discrimination between inventoried landslides and sampled pseudo-absences, **not** the true probability of a future landslide.

**Run order:** Execute all cells from top to bottom. A full sensitivity run may take time, especially on the full 1:2 dataset.


In [1]:
# Optional (run once if packages are missing):
# %pip install geopandas scikit-learn pandas numpy matplotlib xgboost

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedGroupKFold,
    RandomizedSearchCV,
    cross_val_predict,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
OUTER_SPLITS = 5
INNER_SPLITS = 3
METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths and runtime settings

In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_1TO2_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "static_model_1to2.csv"
)

STATIC_1TO1_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "static_model_1to1_sensitivity.csv"
)

TEMPORAL_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "temporal_model_full_enhanced_spatial.csv"
)

WATERWAYS_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Waterways"
)

STAGE08_DIR = (
    DEV_DIR
    / "05_model_results"
    / "enhanced_static"
)

STAGE09_DIR = (
    DEV_DIR
    / "05_model_results"
    / "temporal_incremental"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "stage10_robustness"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# False = fast smoke test; True = complete planned sensitivity experiments.
FULL_RUN = True

# Stage-10 sensitivities use the SAME fixed RF specifications across scenarios.
# These are NEW tests, not replicas of Stage-08 nested tuning.
RF_PARAMS = {
    "n_estimators": 400,
    "max_depth": 7,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

REPEATS = 3 if FULL_RUN else 1
BLOCK_SIZES_DEG = [0.225, 0.45, 0.675] if FULL_RUN else [0.45]

for required_path in [STATIC_1TO2_PATH, TEMPORAL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print("Input static:", STATIC_1TO2_PATH)
print("Input temporal:", TEMPORAL_PATH)
print("Sensitivity output:", OUTPUT_DIR)


Input static: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv
Input temporal: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_full_enhanced_spatial.csv
Sensitivity output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage10_robustness


## 2. Load primary static dataset and review composition

In [3]:
df = pd.read_csv(STATIC_1TO2_PATH)

required_static = [
    "target",
    "longitude",
    "latitude",
    "spatial_block_id",
    "elevation_m",
    "slope_deg",
    "distance_major_fault_m",
]

missing_static = [
    c
    for c in required_static
    if c not in df.columns
]

if missing_static:
    raise ValueError(f"Missing static fields: {missing_static}")

print("Rows:", len(df))
print("Landslides:", int((df["target"] == 1).sum()))
print("Pseudo-absences:", int((df["target"] == 0).sum()))
print("Spatial groups:", df["spatial_block_id"].nunique())

display(
    df["target"].value_counts()
)


Rows: 582
Landslides: 194
Pseudo-absences: 388
Spatial groups: 167


target
0    388
1    194
Name: count, dtype: int64

# PART A — REBUILD THE EXACT STAGE-08 ENHANCED STATIC INPUTS

Stage 08 computed current-waterway distance and engineered static predictors *inside the notebook*, rather than saving a standalone enhanced dataset. The following blocks are copied from Stage 08 to preserve that input construction. Do not silently drop the waterway feature if the local vector is missing: the notebook will stop instead.

In [4]:
def discover_vectors(root):
    extensions = {
        ".shp",
        ".gpkg",
        ".geojson",
    }

    if not root.exists():
        return []

    return [
        p
        for p in root.rglob("*")
        if p.suffix.lower() in extensions
    ]


waterway_files = discover_vectors(
    WATERWAYS_ROOT
)

print("Waterway files found:", len(waterway_files))

for p in waterway_files[:20]:
    print(" -", p)


Waterway files found: 1
 - C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Waterways\gis_osm_waterways_free_1.shp


In [5]:
DISTANCE_CRS = "EPSG:3857"

if waterway_files:

    water_parts = []

    for path in waterway_files:

        try:
            gdf = gpd.read_file(path)

            if len(gdf) == 0 or gdf.crs is None:
                continue

            gdf = gdf.to_crs("EPSG:4326")

            gdf = gdf.loc[
                gdf.geometry.geom_type.isin(
                    [
                        "LineString",
                        "MultiLineString"
                    ]
                )
            ][["geometry"]].copy()

            if len(gdf):
                water_parts.append(gdf)

        except Exception as exc:
            print("Skipped:", path, "|", exc)


    if water_parts:

        waterways = gpd.GeoDataFrame(
            pd.concat(
                water_parts,
                ignore_index=True
            ),
            geometry="geometry",
            crs="EPSG:4326"
        ).to_crs(DISTANCE_CRS)

        water_union = waterways.geometry.unary_union

        sample_points = gpd.GeoDataFrame(
            df.copy(),
            geometry=gpd.points_from_xy(
                df["longitude"],
                df["latitude"]
            ),
            crs="EPSG:4326"
        ).to_crs(DISTANCE_CRS)

        df["distance_waterway_current_m"] = (
            sample_points.geometry.apply(
                lambda p: float(
                    p.distance(water_union)
                )
            ).to_numpy()
        )

        print(
            "Waterway distance available:",
            int(
                df[
                    "distance_waterway_current_m"
                ].notna().sum()
            ),
            "/",
            len(df)
        )

    else:
        df["distance_waterway_current_m"] = np.nan

else:
    df["distance_waterway_current_m"] = np.nan
    print("No local waterway vector found.")


C:\Users\User\AppData\Local\Temp\ipykernel_7692\1587555355.py:44: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  water_union = waterways.geometry.unary_union


Waterway distance available: 582 / 582


In [6]:
df["log_distance_major_fault_m"] = np.log1p(
    pd.to_numeric(
        df["distance_major_fault_m"],
        errors="coerce"
    )
)

df["slope_sq"] = (
    pd.to_numeric(
        df["slope_deg"],
        errors="coerce"
    ) ** 2
)

df["elevation_slope_interaction"] = (
    pd.to_numeric(
        df["elevation_m"],
        errors="coerce"
    )
    *
    pd.to_numeric(
        df["slope_deg"],
        errors="coerce"
    )
)

if df["distance_waterway_current_m"].notna().any():

    df["log_distance_waterway_current_m"] = np.log1p(
        pd.to_numeric(
            df["distance_waterway_current_m"],
            errors="coerce"
        )
    )

else:

    df["log_distance_waterway_current_m"] = np.nan


In [7]:
NUMERIC_FEATURES = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
    "log_distance_major_fault_m",
    "slope_sq",
    "elevation_slope_interaction",
    "distance_waterway_current_m",
    "log_distance_waterway_current_m",
]

CATEGORICAL_FEATURES = [
    c
    for c in [
        "geology_age",
        "geology_unit_type",
    ]
    if c in df.columns
    and df[c].notna().any()
]

missing_features = [
    c
    for c in NUMERIC_FEATURES
    if c not in df.columns or df[c].notna().sum() == 0
]

if missing_features:
    raise ValueError(
        "Stage-08 enhanced predictors unavailable: "
        + str(missing_features)
        + ". Check the waterway vector and original Stage-08 notebook."
    )

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Stage-08 enhanced features:", len(ALL_FEATURES))
print("Categorical:", CATEGORICAL_FEATURES)

pd.DataFrame({
    "feature": ALL_FEATURES,
    "non_null": [int(df[c].notna().sum()) for c in ALL_FEATURES],
    "missing": [int(df[c].isna().sum()) for c in ALL_FEATURES],
}).to_csv(OUTPUT_DIR / "stage10_feature_audit.csv", index=False)


Stage-08 enhanced features: 16
Categorical: ['geology_age', 'geology_unit_type']


# PART B — REUSABLE LEAKAGE-SAFE VALIDATION HELPERS

These helpers use the same 5-fold spatial-group structure as earlier stages. All imputation and encoding stay *within each training fold*. Classification thresholds are selected from the training partition only, using internal spatial out-of-fold predictions. Fixed RF parameters are used for controlled sensitivity experiments; Stage 08/09 remain the authoritative nested-tuning benchmark.

In [8]:
def build_rf_pipeline(
    numeric_features,
    categorical_features
):

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
    ])

    transformers = [
        (
            "num",
            numeric_pipe,
            numeric_features
        )
    ]

    if categorical_features:

        categorical_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore")
            ),
        ])

        transformers.append(
            (
                "cat",
                categorical_pipe,
                categorical_features
            )
        )

    return Pipeline([
        (
            "preprocess",
            ColumnTransformer(transformers)
        ),
        (
            "model",
            RandomForestClassifier(**RF_PARAMS)
        ),
    ])


def get_metrics(
    y_true,
    probability,
    threshold
):

    prediction = (
        np.asarray(probability)
        >= threshold
    ).astype(int)

    result = {
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": np.nan,
        "pr_auc": average_precision_score(y_true, probability),
    }

    if np.unique(y_true).size == 2:
        result["roc_auc"] = roc_auc_score(y_true, probability)

    return result


def choose_threshold(
    y_true,
    probabilities
):

    threshold_rows = []

    for threshold in np.linspace(0.15, 0.85, 141):
        m = get_metrics(y_true, probabilities, threshold)
        threshold_rows.append({
            "threshold": float(threshold),
            "balanced_accuracy": m["balanced_accuracy"],
            "recall": m["recall"],
            "f1": m["f1"],
        })

    ranked = pd.DataFrame(threshold_rows).sort_values(
        ["balanced_accuracy", "recall", "f1"],
        ascending=False
    )

    return float(ranked.iloc[0]["threshold"])


def validate_group_folds(
    dataframe,
    group_column,
    seed=RANDOM_SEED,
    n_splits=OUTER_SPLITS
):

    y = dataframe["target"].astype(int)
    groups = dataframe[group_column].astype(str)

    if groups.nunique() < n_splits:
        raise ValueError("Insufficient spatial groups for folds")

    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    folds = list(splitter.split(dataframe, y, groups))

    for train_idx, test_idx in folds:
        train_groups = set(groups.iloc[train_idx])
        test_groups = set(groups.iloc[test_idx])

        assert train_groups.isdisjoint(test_groups)

        if y.iloc[train_idx].nunique() != 2:
            raise ValueError("Training fold contains one class")

        if y.iloc[test_idx].nunique() != 2:
            raise ValueError(
                "Test fold contains one class. Inspect block allocation "
                "instead of silently changing the split."
            )

    return folds


def evaluate_rf(
    dataframe,
    numeric_features,
    categorical_features,
    group_column,
    experiment,
    seed=RANDOM_SEED,
    outer_folds=None
):

    X = dataframe[numeric_features + categorical_features].copy()
    y = dataframe["target"].astype(int).copy()
    groups = dataframe[group_column].astype(str).copy()

    if outer_folds is None:
        outer_folds = validate_group_folds(
            dataframe,
            group_column,
            seed=seed
        )

    rows = []
    oof_rows = []

    for fold, (train_idx, test_idx) in enumerate(outer_folds, start=1):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        g_train = groups.iloc[train_idx]

        inner = StratifiedGroupKFold(
            n_splits=INNER_SPLITS,
            shuffle=True,
            random_state=seed + fold
        )

        inner_folds = list(inner.split(X_train, y_train, g_train))

        for tr, te in inner_folds:
            if y_train.iloc[tr].nunique() != 2 or y_train.iloc[te].nunique() != 2:
                raise ValueError("Inner fold has only one class")

        pipeline = build_rf_pipeline(
            numeric_features,
            categorical_features
        )

        inner_prob = cross_val_predict(
            clone(pipeline),
            X_train,
            y_train,
            cv=inner_folds,
            method="predict_proba",
            n_jobs=1
        )[:, 1]

        threshold = choose_threshold(y_train, inner_prob)

        pipeline.fit(X_train, y_train)

        train_prob = pipeline.predict_proba(X_train)[:, 1]
        test_prob = pipeline.predict_proba(X_test)[:, 1]

        train_metrics = get_metrics(y_train, train_prob, threshold)
        test_metrics = get_metrics(y_test, test_prob, threshold)

        row = {
            "experiment": experiment,
            "seed": seed,
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "test_positives": int(y_test.sum()),
            "test_controls": int((y_test == 0).sum()),
            "threshold": threshold,
        }

        for metric in METRICS:
            row[f"train_{metric}"] = train_metrics[metric]
            row[f"test_{metric}"] = test_metrics[metric]
            row[f"gap_{metric}"] = train_metrics[metric] - test_metrics[metric]

        rows.append(row)

        for i, test_position in enumerate(test_idx):
            oof_rows.append({
                "experiment": experiment,
                "seed": seed,
                "fold": fold,
                "row_index": int(test_position),
                "target": int(y_test.iloc[i]),
                "probability": float(test_prob[i]),
                "threshold": threshold,
                "validation_group": str(groups.iloc[test_position]),
            })

        print(experiment, "| fold", fold, "| ROC-AUC", round(test_metrics["roc_auc"], 3))

    return pd.DataFrame(rows), pd.DataFrame(oof_rows)


# PART C — ORIGINAL STAGE-08 / STAGE-09 STABILITY AUDIT

Do not confuse *overall out-of-fold ROC-AUC* with the *mean of fold-specific ROC-AUCs*. The latter exposes geographic instability. This section reads the **original** nested spatial benchmark outputs, where available.

In [9]:
stage08_fold_path = STAGE08_DIR / "enhanced_nested_fold_results.csv"
stage09_fold_path = STAGE09_DIR / "temporal_incremental_fold_results.csv"

original_rows = []

for stage_name, folder in [
    ("Stage 08", STAGE08_DIR),
    ("Stage 09", STAGE09_DIR),
]:

    candidate_files = sorted(folder.glob("*fold*.csv"))

    print(stage_name, "fold-file candidates:")
    for path in candidate_files:
        print(" -", path.name)

    for path in candidate_files:
        table = pd.read_csv(path)
        if "test_roc_auc" not in table.columns or "model" not in table.columns:
            continue

        sub = table.copy()
        sub["source_stage"] = stage_name
        sub["source_file"] = path.name
        original_rows.append(sub)

if original_rows:
    original_folds = pd.concat(original_rows, ignore_index=True)
    original_folds.to_csv(
        OUTPUT_DIR / "original_nested_fold_audit.csv",
        index=False
    )

    group_cols = [
        c
        for c in ["source_stage", "source_file", "dataset", "variant", "model"]
        if c in original_folds.columns
    ]

    stability = (
        original_folds.groupby(group_cols, dropna=False)
        .agg(
            n_folds=("test_roc_auc", "count"),
            mean_roc_auc=("test_roc_auc", "mean"),
            std_roc_auc=("test_roc_auc", "std"),
            min_roc_auc=("test_roc_auc", "min"),
            max_roc_auc=("test_roc_auc", "max"),
        )
        .reset_index()
    )

    display(stability)
    stability.to_csv(
        OUTPUT_DIR / "original_spatial_fold_stability.csv",
        index=False
    )
else:
    print("Original fold files not available. Run the following independent RF sensitivity experiments.")


Stage 08 fold-file candidates:
 - enhanced_nested_fold_results.csv
 - enhanced_thresholds_by_fold.csv
Stage 09 fold-file candidates:
 - temporal_incremental_fold_results.csv


,source_stage,source_file,dataset,variant,model,n_folds,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc
0,Stage 08,enhanced_nested_fold_results.csv,NaN,NaN,Logistic Regression,5,0.875033,0.045517,0.820570,0.934295
1,Stage 08,enhanced_nested_fold_results.csv,NaN,NaN,Random Forest,5,0.905382,0.043344,0.860443,0.963462
2,Stage 08,enhanced_nested_fold_results.csv,NaN,NaN,XGBoost,5,0.903359,0.038652,0.860285,0.953846
3,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static + temporal,Logistic Regression,5,0.925323,0.056611,0.835069,0.986968
4,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static + temporal,Random Forest,5,0.940446,0.032171,0.906427,0.985597
5,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static + temporal,XGBoost,5,0.933252,0.032219,0.905382,0.983539
6,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static only,Logistic Regression,5,0.789136,0.096366,0.715278,0.947188
7,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static only,Random Forest,5,0.831127,0.092160,0.733507,0.943759
8,Stage 09,temporal_incremental_fold_results.csv,full_enhanced,Static only,XGBoost,5,0.802800,0.098590,0.678819,0.915879
9,Stage 09,temporal_incremental_fold_results.csv,landcover,Static + temporal,Logistic Regression,5,0.889587,0.065630,0.803333,0.960000


# PART D — PRIMARY ENHANCED STATIC RANDOM FOREST CHECK

This fixed-parameter RF benchmark is a *controlled robustness test*. Its scores may differ from Stage 08, which used nested hyperparameter tuning.

In [10]:
primary_folds, primary_oof = evaluate_rf(
    df,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    "spatial_block_id",
    "static_primary_fixed_rf"
)

primary_folds.to_csv(
    OUTPUT_DIR / "primary_fixed_rf_fold_metrics.csv",
    index=False
)

primary_oof.to_csv(
    OUTPUT_DIR / "primary_fixed_rf_oof.csv",
    index=False
)

display(primary_folds[[
    "fold",
    "test_roc_auc",
    "test_pr_auc",
    "test_balanced_accuracy",
    "gap_roc_auc",
]])


static_primary_fixed_rf | fold 1 | ROC-AUC 0.97
static_primary_fixed_rf | fold 2 | ROC-AUC 0.857
static_primary_fixed_rf | fold 3 | ROC-AUC 0.904
static_primary_fixed_rf | fold 4 | ROC-AUC 0.863
static_primary_fixed_rf | fold 5 | ROC-AUC 0.929


,fold,test_roc_auc,test_pr_auc,test_balanced_accuracy,gap_roc_auc
0,1,0.969551,0.953686,0.878846,0.015556
1,2,0.857278,0.658264,0.861076,0.131521
2,3,0.904306,0.790426,0.817157,0.084914
3,4,0.862611,0.748024,0.771189,0.126671
4,5,0.929255,0.845533,0.849282,0.058131


# PART E — SPATIAL BLOCK SIZE SENSITIVITY

Approximate block widths 0.225°, 0.45°, 0.675° correspond roughly to 25 km, 50 km and 75 km north–south. **These are geographic-degree approximations, not exact equal-km squares**, especially for east–west distances. Group assignment alone does not guarantee a buffer between training and testing points.

In [11]:
spatial_rows = []

for block_deg in BLOCK_SIZES_DEG:

    candidate = df.copy()

    candidate["stage10_block"] = (
        np.floor(candidate["longitude"] / block_deg).astype(int).astype(str)
        + "_"
        + np.floor(candidate["latitude"] / block_deg).astype(int).astype(str)
    )

    label = f"block_{block_deg:.3f}_deg"

    try:
        folds, predictions = evaluate_rf(
            candidate,
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES,
            "stage10_block",
            label
        )
        spatial_rows.append(folds)
    except ValueError as exc:
        print("UNAVAILABLE:", label, "|", exc)
        continue

if spatial_rows:
    spatial_results = pd.concat(spatial_rows, ignore_index=True)
    spatial_results.to_csv(
        OUTPUT_DIR / "spatial_block_size_sensitivity.csv",
        index=False
    )
    display(spatial_results.groupby("experiment")[[
        "test_roc_auc", "test_balanced_accuracy", "gap_roc_auc"
    ]].agg(["mean", "std"]))


block_0.225_deg | fold 1 | ROC-AUC 0.927
block_0.225_deg | fold 2 | ROC-AUC 0.847
block_0.225_deg | fold 3 | ROC-AUC 0.922
block_0.225_deg | fold 4 | ROC-AUC 0.865
block_0.225_deg | fold 5 | ROC-AUC 0.892
block_0.450_deg | fold 1 | ROC-AUC 0.97
block_0.450_deg | fold 2 | ROC-AUC 0.857
block_0.450_deg | fold 3 | ROC-AUC 0.904
block_0.450_deg | fold 4 | ROC-AUC 0.863
block_0.450_deg | fold 5 | ROC-AUC 0.929
block_0.675_deg | fold 1 | ROC-AUC 0.862
block_0.675_deg | fold 2 | ROC-AUC 0.908
block_0.675_deg | fold 3 | ROC-AUC 0.828
block_0.675_deg | fold 4 | ROC-AUC 0.917
block_0.675_deg | fold 5 | ROC-AUC 0.755


test_roc_auc           test_balanced_accuracy            \
                        mean       std                   mean       std   
experiment                                                                
block_0.225_deg     0.890739  0.034942               0.813248  0.058161   
block_0.450_deg     0.904600  0.046981               0.835510  0.042413   
block_0.675_deg     0.853917  0.066127               0.761318  0.073465   

                gap_roc_auc            
                       mean       std  
experiment                             
block_0.225_deg    0.097609  0.035950  
block_0.450_deg    0.083358  0.048558  
block_0.675_deg    0.135150  0.067013

# PART F — PSEUDO-ABSENCE RATIO SENSITIVITY

Load the Stage-06 1:1 model-ready dataset and **rebuild its derived features directly**, using the same equations and waterway geometry as the Stage-08 enhanced static benchmark. No coordinate-based join is needed: multiple records may legitimately have identical coordinates, so coordinates are not safe unique identifiers.

This experiment compares the existing 1:1 and 1:2 samples; it is **not** an independently redrawn pseudo-absence experiment. It may also change the sample locations and validation folds, so interpret performance differences as sensitivity to the existing sample selection and ratio together.

In [12]:
ratio_rows = [primary_folds.assign(ratio="1:2")]

if STATIC_1TO1_PATH.exists():

    ratio_data = pd.read_csv(STATIC_1TO1_PATH)

    # Rebuild the 1:1 enhanced static features independently.
    # Coordinates may repeat across records: do NOT join by coordinates.
    for column in [
        "longitude",
        "latitude",
        "elevation_m",
        "slope_deg",
        "distance_major_fault_m",
        "spatial_block_id",
        "target",
    ]:
        if column not in ratio_data.columns:
            raise ValueError(f"Missing 1:1 required column: {column}")

    ratio_data["log_distance_major_fault_m"] = np.log1p(
        pd.to_numeric(
            ratio_data["distance_major_fault_m"],
            errors="coerce"
        )
    )

    ratio_data["slope_sq"] = (
        pd.to_numeric(
            ratio_data["slope_deg"],
            errors="coerce"
        ) ** 2
    )

    ratio_data["elevation_slope_interaction"] = (
        pd.to_numeric(
            ratio_data["elevation_m"],
            errors="coerce"
        )
        *
        pd.to_numeric(
            ratio_data["slope_deg"],
            errors="coerce"
        )
    )

    # Use the same national waterway geometry as Part A / Stage 08.
    # Recalculate every 1:1 point rather than copying a different row's value.
    if "water_union" not in globals():
        raise ValueError(
            "Waterway geometry is unavailable. Run Part A and verify "
            "the waterway vector files before Part F."
        )

    sample_points_1to1 = gpd.GeoDataFrame(
        ratio_data.copy(),
        geometry=gpd.points_from_xy(
            ratio_data["longitude"],
            ratio_data["latitude"]
        ),
        crs="EPSG:4326"
    ).to_crs(DISTANCE_CRS)

    ratio_data["distance_waterway_current_m"] = (
        sample_points_1to1.geometry.apply(
            lambda p: float(p.distance(water_union))
        ).to_numpy()
    )

    ratio_data["log_distance_waterway_current_m"] = np.log1p(
        ratio_data["distance_waterway_current_m"]
    )

    required_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

    missing_features = [
        column
        for column in required_features
        if column not in ratio_data.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing 1:1 enhanced predictors: " + str(missing_features)
        )

    print("1:1 samples:", len(ratio_data))
    print("1:1 landslides:", int((ratio_data["target"] == 1).sum()))
    print("1:1 controls:", int((ratio_data["target"] == 0).sum()))
    print("1:1 spatial groups:", ratio_data["spatial_block_id"].nunique())

    if not (
        (ratio_data["target"] == 1).sum() == 194
        and (ratio_data["target"] == 0).sum() == 194
    ):
        print(
            "NOTE: 1:1 sample count differs from 194 + 194; "
            "verify Stage-06 selection before interpreting results."
        )

    ratio_result, ratio_oof = evaluate_rf(
        ratio_data,
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES,
        "spatial_block_id",
        "static_ratio_1to1"
    )

    ratio_rows.append(ratio_result.assign(ratio="1:1"))

else:
    print("1:1 dataset not found; skipping 1:1 sensitivity experiment.")

ratio_results = pd.concat(ratio_rows, ignore_index=True)

ratio_results.to_csv(
    OUTPUT_DIR / "pseudoabsence_ratio_sensitivity.csv",
    index=False
)

display(ratio_results.groupby("ratio")[[
    "test_roc_auc", "test_pr_auc", "test_balanced_accuracy"
]].agg(["mean", "std"]))

1:1 samples: 388
1:1 landslides: 194
1:1 controls: 194
1:1 spatial groups: 149
static_ratio_1to1 | fold 1 | ROC-AUC 0.91
static_ratio_1to1 | fold 2 | ROC-AUC 0.821
static_ratio_1to1 | fold 3 | ROC-AUC 0.925
static_ratio_1to1 | fold 4 | ROC-AUC 0.859
static_ratio_1to1 | fold 5 | ROC-AUC 0.831


test_roc_auc           test_pr_auc           test_balanced_accuracy  \
              mean       std        mean       std                   mean   
ratio                                                                       
1:1       0.869263  0.046466    0.830783  0.074990               0.786798   
1:2       0.904600  0.046981    0.799186  0.110195               0.835510   

                 
            std  
ratio            
1:1    0.044458  
1:2    0.042413

# PART G — REPEATED SPATIAL PARTITION SENSITIVITY

Repeat outer group partitioning with multiple random seeds while preserving the same fixed predictors, observations and RF parameters. This tests fold assignment sensitivity, **not** independent inventory data.

In [13]:
repeat_rows = []

for repeat_id in range(REPEATS):

    current_seed = RANDOM_SEED + repeat_id * 101

    folds, predictions = evaluate_rf(
        df,
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES,
        "spatial_block_id",
        f"repeat_{repeat_id + 1}",
        seed=current_seed
    )

    repeat_rows.append(folds)

repeated_results = pd.concat(repeat_rows, ignore_index=True)
repeated_results.to_csv(
    OUTPUT_DIR / "repeated_spatial_partition_sensitivity.csv",
    index=False
)

display(repeated_results.groupby("experiment")[[
    "test_roc_auc", "test_balanced_accuracy", "gap_roc_auc"
]].agg(["mean", "std", "min", "max"]))


repeat_1 | fold 1 | ROC-AUC 0.97
repeat_1 | fold 2 | ROC-AUC 0.857
repeat_1 | fold 3 | ROC-AUC 0.904
repeat_1 | fold 4 | ROC-AUC 0.863
repeat_1 | fold 5 | ROC-AUC 0.929
repeat_2 | fold 1 | ROC-AUC 0.965
repeat_2 | fold 2 | ROC-AUC 0.912
repeat_2 | fold 3 | ROC-AUC 0.852
repeat_2 | fold 4 | ROC-AUC 0.868
repeat_2 | fold 5 | ROC-AUC 0.938
repeat_3 | fold 1 | ROC-AUC 0.975
repeat_3 | fold 2 | ROC-AUC 0.89
repeat_3 | fold 3 | ROC-AUC 0.878
repeat_3 | fold 4 | ROC-AUC 0.9
repeat_3 | fold 5 | ROC-AUC 0.904


test_roc_auc                               test_balanced_accuracy  \
                   mean       std       min       max                   mean   
experiment                                                                     
repeat_1       0.904600  0.046981  0.857278  0.969551               0.835510   
repeat_2       0.907110  0.047208  0.852432  0.965482               0.844638   
repeat_3       0.909421  0.037829  0.878205  0.974684               0.821922   

                                         gap_roc_auc                      \
                 std       min       max        mean       std       min   
experiment                                                                 
repeat_1    0.042413  0.771189  0.878846    0.083358  0.048558  0.015556   
repeat_2    0.070163  0.773484  0.915917    0.081006  0.049356  0.019059   
repeat_3    0.030420  0.796053  0.873892    0.078587  0.039673  0.010039   

                      
                 max  
experiment            
repeat_1    0.131521  
repeat_2    0.137486  
repeat_3    0.111745

# PART H — TEMPORAL FEATURE ABLATION ON IDENTICAL 363-ROW SUBSET

Use the **same full-temporal matched subset** for every ablation. Following Stage 09, static numerical predictors are selected only when present in the source file; missing aspect sine/cosine columns are not fabricated. All ablations retain the same spatial validation folds.


In [14]:
temporal = pd.read_csv(TEMPORAL_PATH)

temporal["target"] = (
    temporal["temporal_role"]
    .eq("case")
    .astype(int)
)

STATIC_TEMP_NUMERIC = [
    "elevation_m",
    "slope_deg",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]

# Match Stage 09: use only static numerical features actually present.
# Stage 09 did not require aspect_sin / aspect_cos in every temporal file.
missing_static = [
    c
    for c in STATIC_TEMP_NUMERIC
    if c not in temporal.columns
]

if missing_static:
    print("Unavailable static temporal predictors:", missing_static)

STATIC_TEMP_NUMERIC = [
    c
    for c in STATIC_TEMP_NUMERIC
    if c in temporal.columns
    and temporal[c].notna().any()
]

if not STATIC_TEMP_NUMERIC:
    raise ValueError("No usable static predictors in Stage-09 temporal dataset")

print("Available Stage-09 static predictors:", STATIC_TEMP_NUMERIC)

STATIC_TEMP_CATEGORICAL = [
    c
    for c in ["geology_age", "geology_unit_type"]
    if c in temporal.columns and temporal[c].notna().any()
]

RAIN_FEATURES = [
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
]

MOISTURE_FEATURES = [
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
]

WATERWAY_FEATURES = [
    "distance_waterway_current_m",
]

LANDCOVER_FEATURES = [
    "landcover_harmonised",
]

required_temporal = (
    STATIC_TEMP_NUMERIC
    + RAIN_FEATURES
    + MOISTURE_FEATURES
    + WATERWAY_FEATURES
    + LANDCOVER_FEATURES
    + ["validation_group", "target"]
)

missing_temporal = [
    c for c in required_temporal
    if c not in temporal.columns
]

if missing_temporal:
    raise ValueError(f"Missing Stage-09 temporal columns: {missing_temporal}")

print("Temporal full subset:", len(temporal))
print("Cases:", temporal["target"].sum())
print("Controls:", (temporal["target"] == 0).sum())
print("Matched validation groups:", temporal["validation_group"].nunique())

# Essential safeguard against splitting a matched case and its controls.
if "matched_group_id" in temporal.columns:
    n_group_mismatch = (
        temporal.groupby("matched_group_id")["validation_group"]
        .nunique().gt(1).sum()
    )
    if n_group_mismatch:
        raise ValueError("Matched groups are split across validation groups")

temporal_folds = validate_group_folds(
    temporal,
    "validation_group"
)

ABLATONS = {
    "static_only": ([], []),
    "full_temporal": (
        RAIN_FEATURES + MOISTURE_FEATURES + WATERWAY_FEATURES,
        LANDCOVER_FEATURES
    ),
    "without_rainfall": (
        MOISTURE_FEATURES + WATERWAY_FEATURES,
        LANDCOVER_FEATURES
    ),
    "without_soil_moisture": (
        RAIN_FEATURES + WATERWAY_FEATURES,
        LANDCOVER_FEATURES
    ),
    "without_landcover": (
        RAIN_FEATURES + MOISTURE_FEATURES + WATERWAY_FEATURES,
        []
    ),
    "without_waterway": (
        RAIN_FEATURES + MOISTURE_FEATURES,
        LANDCOVER_FEATURES
    ),
}

print("Ablation configurations:", list(ABLATONS))


Available Stage-09 static predictors: ['elevation_m', 'slope_deg', 'soil_clay', 'soil_sand', 'soil_silt', 'soil_bulk_density', 'distance_major_fault_m']
Temporal full subset: 363
Cases: 121
Controls: 242
Matched validation groups: 28
Ablation configurations: ['static_only', 'full_temporal', 'without_rainfall', 'without_soil_moisture', 'without_landcover', 'without_waterway']


In [15]:
ablation_rows = []
ablation_oof_rows = []

for name, (extra_numeric, extra_categorical) in ABLATONS.items():

    numeric = STATIC_TEMP_NUMERIC + extra_numeric
    categorical = STATIC_TEMP_CATEGORICAL + extra_categorical

    fold_results, oof_results = evaluate_rf(
        temporal,
        numeric,
        categorical,
        "validation_group",
        name,
        outer_folds=temporal_folds
    )

    ablation_rows.append(fold_results)
    ablation_oof_rows.append(oof_results)

ablation_results = pd.concat(ablation_rows, ignore_index=True)
ablation_oof = pd.concat(ablation_oof_rows, ignore_index=True)

ablation_results.to_csv(
    OUTPUT_DIR / "temporal_ablation_fold_metrics.csv",
    index=False
)

ablation_oof.to_csv(
    OUTPUT_DIR / "temporal_ablation_oof.csv",
    index=False
)

ablation_summary = (
    ablation_results.groupby("experiment")
    .agg(
        mean_roc_auc=("test_roc_auc", "mean"),
        std_roc_auc=("test_roc_auc", "std"),
        mean_pr_auc=("test_pr_auc", "mean"),
        mean_balanced_accuracy=("test_balanced_accuracy", "mean"),
        mean_gap_roc_auc=("gap_roc_auc", "mean"),
    )
    .reset_index()
)

ablation_summary.to_csv(
    OUTPUT_DIR / "temporal_ablation_summary.csv",
    index=False
)

display(ablation_summary.sort_values("mean_roc_auc", ascending=False))


static_only | fold 1 | ROC-AUC 0.752
static_only | fold 2 | ROC-AUC 0.769
static_only | fold 3 | ROC-AUC 0.792
static_only | fold 4 | ROC-AUC 0.964
static_only | fold 5 | ROC-AUC 0.908
full_temporal | fold 1 | ROC-AUC 0.903
full_temporal | fold 2 | ROC-AUC 0.903
full_temporal | fold 3 | ROC-AUC 0.923
full_temporal | fold 4 | ROC-AUC 0.983
full_temporal | fold 5 | ROC-AUC 0.96
without_rainfall | fold 1 | ROC-AUC 0.906
without_rainfall | fold 2 | ROC-AUC 0.906
without_rainfall | fold 3 | ROC-AUC 0.935
without_rainfall | fold 4 | ROC-AUC 0.986
without_rainfall | fold 5 | ROC-AUC 0.941
without_soil_moisture | fold 1 | ROC-AUC 0.93
without_soil_moisture | fold 2 | ROC-AUC 0.899
without_soil_moisture | fold 3 | ROC-AUC 0.92
without_soil_moisture | fold 4 | ROC-AUC 0.979
without_soil_moisture | fold 5 | ROC-AUC 0.95
without_landcover | fold 1 | ROC-AUC 0.907
without_landcover | fold 2 | ROC-AUC 0.838
without_landcover | fold 3 | ROC-AUC 0.862
without_landcover | fold 4 | ROC-AUC 0.97
without_

,experiment,mean_roc_auc,std_roc_auc,mean_pr_auc,mean_balanced_accuracy,mean_gap_roc_auc
4,without_soil_moisture,0.935467,0.030368,0.908890,0.869122,0.057361
3,without_rainfall,0.934914,0.032573,0.907623,0.857740,0.057042
0,full_temporal,0.934265,0.035926,0.905102,0.843669,0.059438
2,without_landcover,0.904684,0.055197,0.830421,0.829881,0.088409
5,without_waterway,0.903776,0.062759,0.863734,0.826137,0.088778
1,static_only,0.837086,0.093806,0.700239,0.696588,0.144127


# PART I — GEOGRAPHIC PERFORMANCE AUDIT

Regional labels are exploratory only. Evaluate ROC-AUC only when a region includes both classes, and display counts rather than treating small regional estimates as conclusive. This uses the primary fixed-RF outer-test predictions.

In [16]:
geo_audit = df.copy().reset_index(drop=True)
geo_audit["row_index"] = geo_audit.index

geo_audit = geo_audit.merge(
    primary_oof[["row_index", "probability", "threshold", "fold"]],
    on="row_index",
    how="left",
    validate="one_to_one"
)

# Use state fields when available; do NOT infer administrative regions from rough longitude cuts.
region_col = next(
    (
        c for c in ["state", "state_name", "negeri", "admin1"]
        if c in geo_audit.columns
    ),
    None
)

geographic_rows = []

if region_col is None:
    print("No state field present. Saving grouped spatial-block breakdown instead.")
    region_col = "spatial_block_id"

for region, sub in geo_audit.groupby(region_col, dropna=False):
    sub = sub.dropna(subset=["probability", "threshold"])

    row = {
        "region": str(region),
        "n_samples": len(sub),
        "n_positives": int(sub["target"].sum()),
        "n_controls": int((sub["target"] == 0).sum()),
        "roc_auc": np.nan,
        "pr_auc": np.nan,
    }

    if sub["target"].nunique() == 2:
        row["roc_auc"] = roc_auc_score(
            sub["target"],
            sub["probability"]
        )
        row["pr_auc"] = average_precision_score(
            sub["target"],
            sub["probability"]
        )

    geographic_rows.append(row)

geographic_results = pd.DataFrame(geographic_rows)
geographic_results.to_csv(
    OUTPUT_DIR / "geographic_error_audit.csv",
    index=False
)

display(geographic_results.sort_values("n_samples", ascending=False).head(25))


No state field present. Saving grouped spatial-block breakdown instead.


,region,n_samples,n_positives,n_controls,roc_auc,pr_auc
29,B_226_7,37,34,3,0.970588,0.997403
22,B_225_9,26,23,3,0.565217,0.867068
28,B_226_6,25,22,3,0.606061,0.912042
19,B_225_6,17,15,2,1.000000,1.000000
0,B_222_11,16,16,0,NaN,NaN
20,B_225_7,16,13,3,0.820513,0.959927
1,B_222_12,14,14,0,NaN,NaN
140,B_259_13,11,8,3,1.000000,1.000000
134,B_258_13,9,7,2,0.857143,0.968254
16,B_225_10,7,4,3,1.000000,1.000000


# PART J — SUMMARIZE ROBUSTNESS AND WRITE DECISION SUPPORT

This stage does not select a final winner automatically. Review geographic stability, repeated partition results, temporal ablation, overfitting and known mapping-coverage constraints before Stage 11.

In [17]:
combined = []

for name, result in [
    ("Primary static RF", primary_folds),
    ("Pseudo-absence ratio", ratio_results),
    ("Repeated spatial partitions", repeated_results),
    ("Spatial block size", spatial_results if spatial_rows else pd.DataFrame()),
    ("Temporal feature ablation", ablation_results),
]:

    if result.empty:
        continue

    group_col = "experiment"
    for experiment, group in result.groupby(group_col):
        combined.append({
            "test_family": name,
            "experiment": experiment,
            "n_folds": len(group),
            "mean_roc_auc": group["test_roc_auc"].mean(),
            "std_roc_auc": group["test_roc_auc"].std(),
            "min_roc_auc": group["test_roc_auc"].min(),
            "mean_pr_auc": group["test_pr_auc"].mean(),
            "mean_balanced_accuracy": group["test_balanced_accuracy"].mean(),
            "mean_train_test_auc_gap": group["gap_roc_auc"].mean(),
        })

robustness_summary = pd.DataFrame(combined)
robustness_summary.to_csv(
    OUTPUT_DIR / "stage10_robustness_summary.csv",
    index=False
)

display(robustness_summary)

study_notes = pd.DataFrame([
    {
        "issue": "Pseudo-absence uncertainty",
        "status": "Not resolved by 1:1 vs 1:2 comparison; separate independent resampling is needed",
    },
    {
        "issue": "Spatial buffer",
        "status": "Spatial block groups applied; no explicit exclusion buffer between train/test blocks",
    },
    {
        "issue": "Temporal causality",
        "status": "Verify event/control reference dates and data cutoffs against Stage-05B audits",
    },
    {
        "issue": "National raster coverage",
        "status": "Not proven; validate completeness of all national raster and categorical features before mapping",
    },
    {
        "issue": "Susceptibility score calibration",
        "status": "Case-control scores are relative; do not interpret as absolute event probabilities",
    },
    {
        "issue": "Hyperparameter tuning",
        "status": "Stage 10 uses fixed RF settings; final comparison should rely on Stage-08/09 nested benchmarks",
    },
])

study_notes.to_csv(
    OUTPUT_DIR / "stage10_methodological_limitations.csv",
    index=False
)

report = {
    "stage": "10",
    "purpose": "Spatial and temporal robustness / sensitivity analysis",
    "model": "Random Forest (fixed sensitivity configuration)",
    "static_samples": int(len(df)),
    "temporal_samples": int(len(temporal)),
    "outer_splits": OUTER_SPLITS,
    "inner_splits_for_threshold": INNER_SPLITS,
    "rf_params": RF_PARAMS,
    "features_static": ALL_FEATURES,
    "completed_outputs": sorted(p.name for p in OUTPUT_DIR.glob("*.csv")),
}

with open(OUTPUT_DIR / "stage10_run_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=4)

print("Stage 10 sensitivity analyses complete.")
print("Outputs:", OUTPUT_DIR)


,test_family,experiment,n_folds,mean_roc_auc,std_roc_auc,min_roc_auc,mean_pr_auc,mean_balanced_accuracy,mean_train_test_auc_gap
0,Primary static RF,static_primary_fixed_rf,5,0.904600,0.046981,0.857278,0.799186,0.835510,0.083358
1,Pseudo-absence ratio,static_primary_fixed_rf,5,0.904600,0.046981,0.857278,0.799186,0.835510,0.083358
2,Pseudo-absence ratio,static_ratio_1to1,5,0.869263,0.046466,0.821053,0.830783,0.786798,0.116176
3,Repeated spatial partitions,repeat_1,5,0.904600,0.046981,0.857278,0.799186,0.835510,0.083358
4,Repeated spatial partitions,repeat_2,5,0.907110,0.047208,0.852432,0.798437,0.844638,0.081006
5,Repeated spatial partitions,repeat_3,5,0.909421,0.037829,0.878205,0.815698,0.821922,0.078587
6,Spatial block size,block_0.225_deg,5,0.890739,0.034942,0.847486,0.779679,0.813248,0.097609
7,Spatial block size,block_0.450_deg,5,0.904600,0.046981,0.857278,0.799186,0.835510,0.083358
8,Spatial block size,block_0.675_deg,5,0.853917,0.066127,0.754690,0.691498,0.761318,0.135150
9,Temporal feature ablation,full_temporal,5,0.934265,0.035926,0.902647,0.905102,0.843669,0.059438


Stage 10 sensitivity analyses complete.
Outputs: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage10_robustness


# How to interpret Stage 10

- **Spatial stability:** Compare mean ± standard deviation and worst-fold ROC-AUC, not just pooled performance.
- **Overfitting:** Large positive train-minus-test gaps suggest poor generalization, but small folds can also create noisy gaps.
- **Ratio sensitivity:** Comparing 1:1 vs 1:2 changes both prevalence and the background sample; PR-AUC/precision can change from the different class balance alone. Prefer balanced accuracy and ROC-AUC for cross-ratio interpretation.
- **Ablation:** `full_temporal` versus other variants is paired on the same 363 samples and folds. Examine *fold-wise* differences, not only the pooled mean.
- **Spatial size:** Different block sizes test geographic sensitivity, not exact 25/50/75 km distance buffers.
- **What is NOT completed:** independent pseudo-absence resampling, full nested retuning for every sensitivity, date-leakage proof, national predictor coverage and final model production training. Do not present these as completed.

## Outputs to send for review

```text
original_spatial_fold_stability.csv
primary_fixed_rf_fold_metrics.csv
spatial_block_size_sensitivity.csv
pseudoabsence_ratio_sensitivity.csv
repeated_spatial_partition_sensitivity.csv
temporal_ablation_summary.csv
temporal_ablation_fold_metrics.csv
geographic_error_audit.csv
stage10_robustness_summary.csv
stage10_methodological_limitations.csv
```

**Next notebook:** `11_final_model_selection.ipynb` — evaluate selection criteria using the authoritative LR/RF/XGBoost nested results and this Stage-10 robustness evidence.
